In [ ]:
#FOR CNH ONLY
import pandas as pd

# Load files
PartsMovement_df = pd.read_excel(
    "pmovdcE CNH 14Aug26.xlsx",
    sheet_name="Non Steron",
    skiprows=4
)

Forecast_df = pd.read_excel(
    "FD CNH Non Steron 14Aug26.xlsx", sheet_name="National"
)

# --- Make P/N case-insensitive ---
PartsMovement_df["P/N"] = PartsMovement_df["P/N"].astype(str).str.upper()

# --- Identify all C- columns ---
c_columns = sorted(
    [col for col in PartsMovement_df.columns if isinstance(col, str) and col.startswith("C-")],
    key=lambda x: int(x.split("-")[1]),
    reverse=True
)

# Required columns
oh_col = "OH"
oo_col = "OO"
dn_price_col = "DN Price"

# --- Aggregation rules ---
aggregation_dict = {col: "sum" for col in c_columns + [oh_col, oo_col]}
aggregation_dict[dn_price_col] = "first"   # take one price per PN

# --- GROUP NATIONALLY BY P/N ONLY ---
df_sum = PartsMovement_df.groupby("P/N", as_index=False).agg(aggregation_dict)

# --- Total Calls ---
df_sum["Total Calls"] = df_sum[c_columns].sum(axis=1)

# --- Drop individual C- columns ---
df_sum = df_sum.drop(columns=c_columns)

# --- Add National identifiers ---
df_sum.insert(0, "Agc", "All")
df_sum.insert(0, "Brc", "National")

# --- Reorder columns ---
df_sum = df_sum[
    ["Brc", "Agc", "P/N", "Total Calls", "DN Price", "OH", "OO"]
]

# --- Sort ---
df_sum = df_sum.sort_values("Total Calls", ascending=False).reset_index(drop=True)


In [ ]:
#Perhitungan RC (Rank Call)
# Sort by Total Calls descending
df_sum = (
    df_sum
    .sort_values(
        by=["Total Calls", "P/N"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# Add cumulative sum
df_sum["Accum."] = df_sum["Total Calls"].cumsum()

# Percentage cumulative
last_accum = df_sum["Accum."].iloc[-1]
df_sum["%Accum."] = (df_sum["Accum."] / last_accum) * 100
df_sum["%Accum."] = df_sum["%Accum."].round(2)

#klasifikasi RC ABCD
def assign_rc(pct, total_calls):
    if total_calls > 0 and pct >= 100:
        return "C"

    if 0 <= pct <= 50:
        return "A"
    elif 50 < pct <= 80:
        return "B"
    elif 80 < pct < 100:
        return "C"
    else:
        return "D"


df_sum["RC"] = df_sum.apply(
    lambda row: assign_rc(row["%Accum."], row["Total Calls"]),
    axis=1
)


In [ ]:
#FOR CNH ONLY
# --- Normalize text for safe filtering & merging ---
Forecast_df["p/n"] = Forecast_df["p/n"].astype(str).str.upper()
Forecast_df["brc"] = Forecast_df["brc"].astype(str).str.upper()
Forecast_df["agc"] = Forecast_df["agc"].astype(str).str.upper()

df_sum["P/N"] = df_sum["P/N"].astype(str).str.upper()

# --- Filter ONLY National + ALL AGC ---
forecast_filtered = Forecast_df[
    (Forecast_df["brc"] == "NATIONAL") &
    (Forecast_df["agc"] == "ALL AGC")
]

# --- Keep only P/N + FD_final ---
forecast_subset = forecast_filtered[["p/n", "FD_final"]].rename(
    columns={"p/n": "P/N"}
)

# --- Merge into df_sum ---
df_sum = df_sum.merge(
    forecast_subset,
    on="P/N",
    how="left"
)

In [ ]:
#Urutan Kolom df_sum
# --- Define your desired final column order ---
final_columns = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "OH",
    "OO",
    "Accum.",
    "%Accum."
]

# --- Reorder df_sum (only keep those that exist) ---
df_sum = df_sum[final_columns]

In [ ]:
#Perhitungan Max
# RC → multiplier
rc_multiplier = {
    "A": 5,
    "B": 4,
    "C": 2.5,
    "D": 0
}

# Convert RC class into numeric multiplier
df_sum["RC_value"] = df_sum["RC"].map(rc_multiplier)
df_sum["Max"] = df_sum["FD_final"] * df_sum["RC_value"]
df_sum["Max"] = df_sum["Max"].round(0)
df_sum = df_sum.drop(columns=["RC_value"])

#Urutan Kolom df_sum
# --- Define your desired final column order ---
final_columns = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "Max",
    "OH",
    "OO",
    "Accum.",
    "%Accum."
]

# --- Reorder df_sum (only keep those that exist) ---
df_sum = df_sum[final_columns]

In [ ]:
#INCOMING
# List of new incoming columns
incoming_cols = [f"Incoming M-{i}" for i in range(1, 8)]

# Insert after OO
oo_index = df_sum.columns.get_loc("OO")

for i, col_name in enumerate(incoming_cols):
    if col_name not in df_sum.columns:        # <-- Prevent duplicate error
        df_sum.insert(oo_index + 1 + i, col_name, "")


In [ ]:
#Estimated OH & Estimated OO
import numpy as np

# ----- CREATE COLUMN NAMES -----
est_oh_cols = [f"Estimated OH M-{i}" for i in range(1, 10)]   # M-1 to M-9
est_oo_cols = [f"Estimated OO M-{i}" for i in range(1, 9)]    # M-1 to M-8

# Insert Estimated OH columns after Incoming M-7
insert_pos = df_sum.columns.get_loc("Incoming M-7") + 1
for i, col in enumerate(est_oh_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# Insert Estimated OO columns after Estimated OH M-9
insert_pos = df_sum.columns.get_loc("Estimated OH M-9") + 1
for i, col in enumerate(est_oo_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE VALUES ROW-BY-ROW -----
for idx, row in df_sum.iterrows():

    # Get fixed values
    OH0 = row["OH"]
    OO0 = row["OO"]
    FD = row["FD_final"]
    Max = row["Max"]

    # Store results
    est_oh = {}
    est_oo = {}

    # ----- Estimated OH M-1 -----
    incoming_1 = float(row["Incoming M-1"]) if row["Incoming M-1"] not in ["", None, np.nan] else 0
    est_oh[1] = (OH0 + OO0 + incoming_1) - FD

    # ----- Estimated OO M-1 -----
    est_oo[1] = 0 if est_oh[1] > Max else (Max - est_oh[1])

    # ----- M-2 to M-9 for Estimated OH, M-2 to M-8 for Estimated OO -----
    for i in range(2, 10):

        incoming_i = (
            float(row[f"Incoming M-{i}"])
            if (f"Incoming M-{i}" in df_sum.columns and row[f"Incoming M-{i}"] not in ["", None, np.nan])
            else 0
        )

        # Estimated OH M-i
        prev_oh = est_oh[i-1]
        prev_oo = est_oo[i-1] if i-1 in est_oo else 0
        est_oh[i] = (prev_oh + incoming_i + prev_oo) - FD

        # Estimated OO only until M-8
        if i <= 8:
            est_oo[i] = 0 if est_oh[i] > Max else (Max - est_oh[i])

    # ----- Assign to dataframe -----
    for i in range(1, 10):
        df_sum.loc[idx, f"Estimated OH M-{i}"] = est_oh[i]

    for i in range(1, 9):
        df_sum.loc[idx, f"Estimated OO M-{i}"] = est_oo[i]


In [ ]:
#Schedule Order
import numpy as np

# ----- CREATE COLUMN NAMES -----
schedule_cols = [f"Schedule Order M-{i}" for i in range(1, 7)]

# Insert Schedule Order columns after Estimated OO M-8 (the last OO column)
insert_pos = df_sum.columns.get_loc("Estimated OO M-8") + 1
for i, col in enumerate(schedule_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE SCHEDULE ORDER -----
for idx, row in df_sum.iterrows():

    Max = row["Max"]

    for i in range(1, 7):

        # Lookahead month: OH M-(i+3)
        oh_col = f"Estimated OH M-{i+3}"

        # If OH value exists, fetch it, else use 0
        oh_value = row[oh_col] if oh_col in df_sum.columns else 0

        # Schedule Order formula
        if oh_value > Max:
            sched = 0
        else:
            sched = Max - oh_value

        df_sum.loc[idx, f"Schedule Order M-{i}"] = sched

In [ ]:
#Amount Schedule Order
import numpy as np

# ----- CREATE COLUMN NAMES -----
amount_cols = [f"Amount Schedule Order M-{i}" for i in range(1, 7)]

# Find the position after "Schedule Order M-6"
insert_pos = df_sum.columns.get_loc("Schedule Order M-6") + 1

# Insert empty columns first (if not already present)
for i, col in enumerate(amount_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE AMOUNTS -----
for idx, row in df_sum.iterrows():

    dn_price = row["DN Price"]

    for i in range(1, 7):

        sched_col = f"Schedule Order M-{i}"
        amount_col = f"Amount Schedule Order M-{i}"

        schedule_value = row[sched_col] if sched_col in df_sum.columns else 0

        df_sum.loc[idx, amount_col] = schedule_value * dn_price
        # Round ONLY the Amount Schedule Order columns
for col in amount_cols:
    df_sum[col] = df_sum[col].round(2)

In [ ]:
#FOR CNH ONLY
desc_df = (
    PartsMovement_df[["P/N", "Desc"]]
    .dropna(subset=["Desc"])
    .assign(len_desc=lambda x: x["Desc"].str.len())
    .sort_values("len_desc", ascending=False)
    .drop_duplicates("P/N")[["P/N", "Desc"]]
)
df_sum = df_sum.merge(desc_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("Amount Schedule Order M-6") + 1
desc_series = df_sum.pop("Desc")
df_sum.insert(insert_pos, "Desc", desc_series)


In [ ]:
#MM06
c06_cols = [f"C-{i}" for i in range(1, 7)]  # C-1 to C-6
c06_cols = [col for col in c06_cols if col in PartsMovement_df.columns]  # verify exist
# Group by P/N and sum first
temp = PartsMovement_df.groupby("P/N", as_index=False)[c06_cols].sum()

# Count how many months have calls > 0
temp["MM06"] = temp[c06_cols].gt(0).sum(axis=1)

# Keep only P/N + MM06
mm06_df = temp[["P/N", "MM06"]]
df_sum = df_sum.merge(mm06_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("Total Calls") + 1
mm06_series = df_sum.pop("MM06")
df_sum.insert(insert_pos, "MM06", mm06_series)

In [ ]:
#MM12
c12_cols = [f"C-{i}" for i in range(1, 13)]  # C-1 to C-12
c12_cols = [col for col in c12_cols if col in PartsMovement_df.columns]  # ensure exist
temp12 = PartsMovement_df.groupby("P/N", as_index=False)[c12_cols].sum()

# Count how many months have calls > 0 (non-zero values)
temp12["MM12"] = temp12[c12_cols].gt(0).sum(axis=1)

# Keep only P/N + MM12
mm12_df = temp12[["P/N", "MM12"]]
df_sum = df_sum.merge(mm12_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("MM06") + 1
mm12_series = df_sum.pop("MM12")
df_sum.insert(insert_pos, "MM12", mm12_series)

In [ ]:
#Amount Estimated OO
est_oo_cols = [col for col in df_sum.columns if col.startswith("Estimated OO M-")]
est_oo_cols = sorted(
    est_oo_cols, 
    key=lambda x: int(x.split("M-")[1])  # ensure correct order M-1, M-2, ...
)
# Create monetary columns: DN Price × Estimated OO M-i
for col in est_oo_cols:
    month_num = col.split("M-")[1]  # extracts "1", "2", ... "8"
    new_col = f"Amount Estimated OO M-{month_num}"
    df_sum[new_col] = df_sum["DN Price"] * df_sum[col]
# Find insertion point (after MM12)
insert_pos = df_sum.columns.get_loc("MM12") + 1

# Collect new amount columns in the same order
est_oo_amount_cols = [f"Amount Estimated OO M-{i}" for i in range(1, 9)]

# Move them into correct position
for i, col in enumerate(est_oo_amount_cols):
    series = df_sum.pop(col)
    df_sum.insert(insert_pos + i, col, series)


In [ ]:
# --- RENAME COLUMNS FIRST ---

# Create a rename dictionary
rename_dict = {}

for col in df_sum.columns:
    new_col = col
    new_col = new_col.replace("Estimated", "Est.")
    new_col = new_col.replace("Amount", "Amt.")
    new_col = new_col.replace("Schedule","Sched.")
    rename_dict[col] = new_col

# Apply renaming
df_sum = df_sum.rename(columns=rename_dict)


# --- NOW REBUILD THE COLUMN ORDER USING UPDATED NAMES ---

# STARTING COLUMNS
desired_order_start = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "Max",
    "OH",
    "OO"
]

# Dynamic column groups (renamed ones)
incoming_cols = [col for col in df_sum.columns if col.startswith("Incoming M-")]
est_oh_cols   = [col for col in df_sum.columns if col.startswith("Est. OH M-")]
est_oo_cols   = [col for col in df_sum.columns if col.startswith("Est. OO M-")]
schedule_cols = [col for col in df_sum.columns if col.startswith("Sched. Order M-")]
amt_cols      = [col for col in df_sum.columns if col.startswith("Amt. Sched. Order M-") or col.startswith("Amt. Est. OO")]

ending_cols = [
    "Desc",
    "MM06",
    "MM12",
    "Trend Coef",   # add this
    "Accum.",
    "%Accum."
]

# Final order (only include columns that exist)
final_order = (
    desired_order_start
    + incoming_cols
    + est_oh_cols
    + est_oo_cols
    + schedule_cols
    + amt_cols
    + ending_cols
)

final_order = [col for col in final_order if col in df_sum.columns]

# Apply reordering
df_sum = df_sum[final_order]


In [ ]:
#TREND COEFFICIENT
import numpy as np

# --- STEP 1: Identify only D-1 to D-12 ---
d_cols = [
    col for col in PartsMovement_df.columns
    if isinstance(col, str) and col.startswith("D-") and col[2:].isdigit() and 1 <= int(col[2:]) <= 12
]

# Sort numerically: D-1, D-2, ... D-12
d_cols = sorted(d_cols, key=lambda x: int(x.split("-")[1]))

# Reverse to chronological order: D-12 → D-1
d_cols_reversed = list(reversed(d_cols))

# --- STEP 2: Compute Trend Coef (slope) ---
trend_list = []

for pn, group in PartsMovement_df.groupby("P/N"):
    # Sum all branch values for this PN
    values = group[d_cols_reversed].astype(float).sum(axis=0).values
    
    x = np.arange(len(values))  # 0..11 timeline
    
    slope = np.polyfit(x, values, 1)[0]  # linear regression slope
    
    trend_list.append([pn, slope])

trend_df = pd.DataFrame(trend_list, columns=["P/N", "Trend Coef"])

# --- STEP 3: Merge into df_sum ---
df_sum = df_sum.merge(trend_df, on="P/N", how="left")

# --- STEP 4: Insert Trend Coef after MM12 ---
insert_pos = df_sum.columns.get_loc("MM12") + 1
trend_series = df_sum.pop("Trend Coef")
df_sum.insert(insert_pos, "Trend Coef", trend_series)


In [ ]:
# =========================
# MERGE ALERT COLUMNS
# =========================

alert_filtered = Forecast_df[
    (Forecast_df["brc"] == "NATIONAL") &
    (Forecast_df["agc"] == "ALL AGC")
][["p/n", "forecast_alert_score", "forecast_alert_label"]]

df_sum = df_sum.merge(
    alert_filtered,
    left_on="P/N",
    right_on="p/n",
    how="left"
)

df_sum = df_sum.drop(columns=["p/n"])
# =========================
# ADD ADJUSTMENT COLUMNS
# BEFORE forecast_alert_score
# =========================

insert_position = df_sum.columns.get_loc("forecast_alert_score")

new_columns = [
    "Chk. OO",
    "Adj OO M-1",
    "Adj OO M-2",
    "Adj OO M-3",
    "Adj Sch.Ord M-1",
    "Adj Sch.Ord M-2",
    "Adj Sch.Ord M-3"
]

for idx, col_name in enumerate(new_columns):

    df_sum.insert(insert_position + idx, col_name, 0)

In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


# =========================
# EXPORT DATAFRAME FIRST
# =========================

output_path = "FD_Processed_Output.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_sum.to_excel(writer, index=False, sheet_name="FD Processed")

# =========================
# OPEN EXCEL FILE AGAIN
# =========================

wb = load_workbook(output_path)
ws = wb["FD Processed"]  # sheet names are case-sensitive

# =========================
# CREATE HEADER DICTIONARY
# =========================

headers = {}

for cell in ws[1]:
    headers[cell.value] = cell.column

print(headers.keys())

# =========================
# WRITE FORMULAS ROW BY ROW
# =========================

for row in range(2, ws.max_row + 1):

    # ---------------------------------
    # FIXED COLUMNS
    # ---------------------------------

    oh_col = get_column_letter(headers["OH"])
    oo_col = get_column_letter(headers["OO"])
    fd_col = get_column_letter(headers["FD_final"])
    max_col = get_column_letter(headers["Max"])
    price_col = get_column_letter(headers["DN Price"])
    chk_oo_col = get_column_letter(headers["Chk. OO"])
    
    # =========================================
    # CHK. OO
    # =========================================

    ws[f"{chk_oo_col}{row}"] = (
        f'=IF({oo_col}{row}>=5*{max_col}{row},"Y","N")'
    )

    # =========================================
    # ESTIMATED OH + ESTIMATED OO
    # =========================================

    for i in range(1, 10):

        est_oh_col = get_column_letter(headers[f"Est. OH M-{i}"])

        # =====================================
        # INCOMING EXISTS ONLY UNTIL M-7
        # =====================================

        if i <= 7:
            incoming_col = get_column_letter(headers[f"Incoming M-{i}"])

        # =====================================
        # M-1
        # =====================================

        if i == 1:

            ws[f"{est_oh_col}{row}"] = (
                f"=({oh_col}{row}+{oo_col}{row}+{incoming_col}{row})-{fd_col}{row}"
            )

        # =====================================
        # M-2 UNTIL M-9
        # =====================================

        else:

            prev_est_oh_col = get_column_letter(headers[f"Est. OH M-{i-1}"])
            prev_est_oo_col = get_column_letter(headers[f"Est. OO M-{i-1}"])

            # ---------------------------------
            # IF INCOMING EXISTS
            # ---------------------------------

            if i <= 7:

                ws[f"{est_oh_col}{row}"] = (
                    f"=({prev_est_oh_col}{row}+{prev_est_oo_col}{row}+{incoming_col}{row})-{fd_col}{row}"
                )

            # ---------------------------------
            # IF NO INCOMING EXISTS
            # ---------------------------------

            else:

                ws[f"{est_oh_col}{row}"] = (
                    f"=({prev_est_oh_col}{row}+{prev_est_oo_col}{row})-{fd_col}{row}"
                )

        # =====================================
        # ESTIMATED OO
        # =====================================

        if i <= 8:

            est_oo_col = get_column_letter(headers[f"Est. OO M-{i}"])

            # ---------------------------------
            # M-1 UNTIL M-3 WITH ADJUSTMENT
            # ---------------------------------

            if i <= 3:

                adjusted_oo_col = get_column_letter(headers[f"Adj OO M-{i}"])

                ws[f"{est_oo_col}{row}"] = (
                    f'=IF({chk_oo_col}{row}="Y",'
                    f'IF({est_oh_col}{row}>{max_col}{row},0,{max_col}{row}-{est_oh_col}{row})+{adjusted_oo_col}{row},'
                    f'IF({est_oh_col}{row}>{max_col}{row},0,{max_col}{row}-{est_oh_col}{row}))'
                )

            # ---------------------------------
            # M-4 UNTIL M-8 NORMAL
            # ---------------------------------

            else:

                ws[f"{est_oo_col}{row}"] = (
                    f"=IF({est_oh_col}{row}>{max_col}{row},0,{max_col}{row}-{est_oh_col}{row})"
                )

    # =========================================
    # SCHEDULE ORDER
    # =========================================

    for i in range(1, 7):

        sched_col = get_column_letter(headers[f"Sched. Order M-{i}"])

        future_est_oh_col = get_column_letter(headers[f"Est. OH M-{i+3}"])

        # ---------------------------------
        # M-1 UNTIL M-3 WITH ADJUSTMENT
        # ---------------------------------

        if i <= 3:

            adj_sched_col = get_column_letter(headers[f"Adj Sch.Ord M-{i}"])

            ws[f"{sched_col}{row}"] = (
                f"=IF({future_est_oh_col}{row}>{max_col}{row},0,{max_col}{row}-{future_est_oh_col}{row})+{adj_sched_col}{row}"
            )

        # ---------------------------------
        # M-4 UNTIL M-6 NORMAL
        # ---------------------------------

        else:

            ws[f"{sched_col}{row}"] = (
                f"=IF({future_est_oh_col}{row}>{max_col}{row},0,{max_col}{row}-{future_est_oh_col}{row})"
            )
            
    # =========================================
    # AMOUNT ESTIMATED OO COLUMNS
    # =========================================

    for i in range(1, 9):

        est_oo_col = get_column_letter(headers[f"Est. OO M-{i}"])
        amt_est_oo_col = get_column_letter(headers[f"Amt. Est. OO M-{i}"])

        ws[f"{amt_est_oo_col}{row}"] = (
            f"={est_oo_col}{row}*{price_col}{row}"
        )

    # =========================================
    # AMOUNT COLUMNS
    # =========================================

    for i in range(1, 7):

        sched_col = get_column_letter(headers[f"Sched. Order M-{i}"])
        amt_col = get_column_letter(headers[f"Amt. Sched. Order M-{i}"])

        ws[f"{amt_col}{row}"] = (
            f"={sched_col}{row}*{price_col}{row}"
        )

# =========================
# SAVE FILE
# =========================

wb.save(output_path)

print(f"File exported successfully: {output_path}")